# Chapter 17 — Raw Output First

**Companion to Applied AI**

Question: Why must the observation outlive its interpretation?

By the end of this notebook you will have:

- stored raw provider bytes before parsing
- reprocessed them with a v2 parser, appending a new reading
- shown reinterpretation refused when bytes are missing

## What this notebook demonstrates
`preserve → interpret` against `interpret → discard`: a v2 parser rescues a day-1 success that v1 misread — with zero new provider calls.

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
import json

seed: 42


## 1. Store raw bytes first, parse second

In [2]:
archive = {}  # call_id -> raw bytes
def observe(call_id: str, raw: bytes):
    archive[call_id] = raw  # bytes land before any parser runs

observe("c-day1", b'{"verdict": "complete", "reason": "looks done to me"}')
print("archived:", archive)

archived: {'c-day1': b'{"verdict": "complete", "reason": "looks done to me"}'}


## 2. v1 reads text; v2 demands a reason code — same bytes, new reading

In [3]:
def interpret_v1(raw: bytes):
    d = json.loads(raw)
    return {"status": "succeeded" if d.get("verdict") == "complete" else "unresolved"}

def interpret_v2(raw: bytes):
    d = json.loads(raw)
    if "reason_code" not in d:
        return {"status": "unresolved", "why": "v2 requires reason_code"}
    return {"status": "succeeded", "code": d["reason_code"]}

r1 = interpret_v1(archive["c-day1"])
r2 = interpret_v2(archive["c-day1"])
print("v1:", r1, "\nv2:", r2)
assert r1["status"] == "succeeded" and r2["status"] == "unresolved"
print("day-1 'success' becomes day-2 'unresolved' — without spending a new receipt")

v1: {'status': 'succeeded'} 
v2: {'status': 'unresolved', 'why': 'v2 requires reason_code'}
day-1 'success' becomes day-2 'unresolved' — without spending a new receipt


## 3. Break it: lose the bytes, lose the right to reinterpret

In [4]:
def reinterpret(call_id: str, parser):
    if call_id not in archive:
        raise LookupError(f"ObservationUnavailable: no bytes for {call_id}")
    return parser(archive[call_id])

del archive["c-day1"]
try:
    reinterpret("c-day1", interpret_v2)
    raise SystemExit("should have refused")
except LookupError as e:
    print("refused as required:", e)

refused as required: ObservationUnavailable: no bytes for c-day1


## Interpretation
- Supports: observations must outlive interpretations; reinterpretation appends versioned readings with zero new receipts.
- Does NOT support: claims about any real provider log format.

## Try it yourself
1. Add `reason_code` to the raw bytes and watch v2 succeed.
2. Corrupt one byte and show v1 raising while the archive still holds evidence.
3. Keep both readings side by side with parser-version tags.